# Retrieval – zadania

## Vector store, retriever i RAG

Zbuduj RAG-a odpowiadającego na pytania o "Pana Tadeusza" i zapisz jego wyniki do pliku – później posłużą do ewaluacji.

**Kroki:**
1. Połącz się z kolekcją `pan_tadeusz` (`QdrantVectorStore`) i utwórz z niej retriever.
2. Dla kilku pytań z listy `questions` (poniżej) sprawdź, jakie chunki zwraca retriever. Potestuj różne wartości `k`.
3. Napisz funkcję `rag()` (2-step RAG), która dla pytania pobiera chunki, buduje z nich prompt i zwraca odpowiedź modelu razem z użytym kontekstem. W prompcie poproś model, aby odpowiadał wyłącznie na podstawie kontekstu, a jeśli nie ma w nim odpowiedzi – odpowiedział „Nie wiem”.
4. Odpowiedz na wszystkie pytania i zapisz wyniki do pliku `result.json` jako listę słowników:

```json
[
    {
        "model_input": "pytanie",
        "expected_output": "poprawna odpowiedź (z listy expected_answers)",
        "actual_output": "odpowiedź RAG-a",
        "retrieval_context": ["treść chunka 1", "treść chunka 2", "..."]
    },
    ...
]
```

5. Przejrzyj odpowiedzi: na które pytania RAG odpowiedział poprawnie, a gdzie zabrakło właściwego chunka w kontekście?

**Dla chętnych:** zadaj 2–3 pytania agentowi (`create_agent` + narzędzie do wyszukiwania w kolekcji `pan_tadeusz`) i porównaj, ile razy i z jakimi zapytaniami korzystał z narzędzia.

---
(czas: 20 min.)

In [ ]:
questions = [
    "Pomiędzy kim trwa spór o zamek Horeszków?",
    "Jak miał na imię kapitan Ryków?",
    "Wymień wszystkie zioła które znajdowały się w donicach na oknach w domu Tadeusza.",
    "'Kapitanie! nie splamisz czci twojej pardonem. Dałeś próby, rycerzu nieszczęsny, lecz mężny' – kto wypowiada te słowa?",
    "Od kogo Sędzia otrzymał w darze kryształowy kufel?",
    "Czym się różnili Dobrzyńscy od reszty Litwinów?",
    "Co mówi Sędzia o wódce gdańskiej?",
    "Jak Domejko i Dowejko nazwali karczmę którą zbudowali po zakończeniu sporu?",
    "Kto wypowiedział słowa 'Kometa czasem wojny, czasem wróży kłótnie!'",
    "Jakie ustawy znajdują się w żółtej księdze?"
]

expected_answers = [
    "Spór o zamek trwał między rodami Sopliców i Horeszków.",
    "Kapitan Ryków miał na imię Nikita.",
    "Te zioła to: Geranium, lewkonija, astry i fijołki.",
    "Te słowa wypowiada Podkomorzy.",
    "Dostał go od księdza Robaka.",
    "Dobrzyńscy wyróżniali się językiem oraz wzrostem.",
    "Mówi, że 'Wódka gdańska to napój miły dla Polaka'.",
    "Nazwali tę karczmę 'Niedźwiadek'.",
    "Te słowa wypowiedział Wojski.",
    "W żółtej księdze znajdują się ustawy wojenne."
]

In [ ]:
# ...

## Filtrowanie po payloadzie

Sprawdź, jak filtrowanie po metadanych wpływa na trafność kontekstu i odpowiedzi.

**Kroki:**
1. Utwórz w kolekcji `pan_tadeusz` indeksy payloadu dla pól `metadata.book_number` i `metadata.chunk_length` (typ `INTEGER`).
2. Rozszerz funkcję `rag()` tak, aby przyjmowała dodatkowe parametry wyszukiwania i przekazywała je do retrievera.
3. Odpowiedz na poniższe pytania – najpierw bez filtra, a potem z filtrem – i porównaj kontekst oraz odpowiedzi:

| Pytanie | Filtr |
|---|---|
| Jak miał na imię kapitan Ryków? | tylko księga X (`MatchValue`) |
| Od kogo Sędzia otrzymał w darze kryształowy kufel? | księgi I–VI (`Range`) |
| Co mówi Sędzia o wódce gdańskiej? | księgi III, IV lub V (`MatchAny`) |
| Wymień wszystkie zioła które znajdowały się w donicach na oknach w domu Tadeusza. | księga I **oraz** chunki dłuższe niż 300 znaków (dwa warunki w `must`) |

4. Odpowiedz na jedno wybrane pytanie z filtrem wskazującym **niewłaściwą** księgę. Jak zachowuje się model, gdy w kontekście nie ma odpowiedzi?

---
(czas: 18 min.)

In [ ]:
# ...

## Hybrid search

Utwórz hybrydową wersję kolekcji z "Panem Tadeuszem" i porównaj wyszukiwanie dense, sparse (BM25) i hybrid.

**Kroki:**
1. Przygotuj chunki tak jak w zadaniu z kolekcją `pan_tadeusz`: wczytaj osobno każdy z 13 plików PDF z folderu `documents/pan-tadeusz/` (`PyMuPDFLoader`), podziel je `RecursiveCharacterTextSplitter` (`chunk_size=1000`, `chunk_overlap=200`) i uzupełnij metadane o `book_number` i `chunk_length` (pozostaw też `title` i `page`).
2. Utwórz kolekcję `pan_tadeusz_hybrid` z wektorami dense (`"dense"`) i sparse (`"bm25"`, z modyfikatorem IDF).
3. Utwórz `QdrantVectorStore` w trybie `RetrievalMode.HYBRID` i dodaj do niego chunki metodą `add_documents()`.
4. Dla poniższych pytań porównaj wyniki trzech trybów wyszukiwania (`DENSE`, `SPARSE`, `HYBRID`) – na którym miejscu jest chunk zawierający odpowiedź?
   - `"Jak miał na imię kapitan Ryków?"`
   - `"Jak Domejko i Dowejko nazwali karczmę którą zbudowali po zakończeniu sporu?"`
   - `"Kto wypowiedział słowa 'Kometa czasem wojny, czasem wróży kłótnie!'"`
5. Odpowiedz na wszystkie pytania z listy `questions` RAG-iem z hybrydowym retrieverem i porównaj odpowiedzi z wynikami z pierwszego zadania.

Przy ocenie wyników pamiętaj, że BM25 dopasowuje słowa dokładnie – nie zna polskiej odmiany.

---
(czas: 20 min.)

In [ ]:
# ...

## Reranking

Dodaj reranking do retrievera. Ustaw `top_k` w retrieverze na stosunkowo dużą wartość a w rerankingu wybierz mniejszą.

**Kroki:**
1. Utwórz retriever, który pobiera 10 chunków (`k=10`) – z kolekcji `pan_tadeusz` albo `pan_tadeusz_hybrid`.
2. Przygotuj dwa rerankery, które zostawiają 3 najlepsze chunki:
   - LLM-owy – `LLMListwiseRerank`,
   - cross-encoder – model `sdadas/polish-reranker-roberta-v3`.
3. Dla 3 wybranych pytań z listy `questions` porównaj kontekst w trzech wariantach: bez rerankingu (3 pierwsze chunki z retrievera), z rerankerem LLM-owym i z cross-encoderem. Który wariant najczęściej umieszcza chunk z odpowiedzią na pierwszym miejscu?
4. Odpowiedz na te same pytania RAG-iem z rerankingiem i porównaj odpowiedzi z wersją bez rerankingu.

---
(czas: 15 min.)

In [ ]:
# ...